In [ ]:
# Este ejemplo muestra los CUATRO cuadrantes posibles

# 1. Alta cohesión + Alto acoplamiento (frecuente en código legacy)
class ReportePóliza:
    def generar(self, poliza_id: str):
        # Cohesión OK: todo es sobre reportes de póliza
        # Acoplamiento MALO: construye directamente sus dependencias
        db = ConexionPostgres(host="10.0.0.1", port=5432)   # acoplado a Postgres
        pdf = GeneradorPDFv2()                               # acoplado a versión específica
        datos = db.query(f"SELECT * FROM polizas WHERE id={poliza_id}")
        return pdf.renderizar(datos)


# 2. Alta cohesión + Bajo acoplamiento (el objetivo)
class ReportePóliza:
    def __init__(self, repositorio: RepositorioPólizas, generador: GeneradorDocumento):
        self.repositorio = repositorio      # contrato, no implementación
        self.generador = generador          # contrato, no implementación

    def generar(self, poliza_id: str):
        datos = self.repositorio.obtener(poliza_id)
        return self.generador.renderizar(datos)
    # Cohesión alta: solo sabe de reportes
    # Acoplamiento bajo: no sabe si es Postgres, MySQL, PDF, Word...


# 3. Baja cohesión + Bajo acoplamiento (módulos genéricos e inútiles)
class Procesador:
    def __init__(self, servicio: Servicio):
        self.servicio = servicio

    def ejecutar_a(self): ...   # ¿qué hace esto?
    def ejecutar_b(self): ...   # sin relación con lo anterior
    def ejecutar_c(self): ...   # imposible de testear o razonar


# 4. Baja cohesión + Alto acoplamiento (el peor escenario — God Class)
class Gestor:
    def procesar_siniestro(self): ...
    def enviar_email(self): ...
    def conectar_bd(self): ...
    def calcular_impuesto(self): ...
    # Hace todo, sabe de todo — el cambio en cualquier parte lo rompe